# Physiology: discover stationary luminous pulses — Bocas 2026

Search native-resolution vessel regions for nearly stationary light that brightens and fades. Outputs are **unconfirmed candidates**, ranked for visual review. Fiber-like events stay in the tables with flags. Start with the local pilot; the Box batch below is deliberately small and spread across recording times.

The initial regions enclose **vessel groups**, not verified individual vessel outlines. They were estimated from the local reference's temporal maximum projection. Inspect the overlay and edit `REGIONS` using a brighter frame when available. Replace each group with separate, non-overlapping vessel polygons for more selective illumination rejection. Coordinates are original full side-by-side video pixels; each camera is analyzed independently.

Use the `manta` Python environment. The existing brightness notebook remains available for comparison.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import sys, json, re, tempfile, time
import urllib.error, urllib.parse, urllib.request
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Video, HTML, FileLink

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scripts' / 'physiologyDiscovery.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
import physiologyDiscovery as discovery
import brightnessOverTime as brightness

TIMEZONE = 'America/Panama'
NIGHT_BOUNDARY_HOUR = 12
REFERENCE_VIDEO = PROJECT_ROOT / 'data/selected_videos/physiology/sbs_20260828_235910.mkv'
RESULTS_ROOT = PROJECT_ROOT / 'data/processed/physiology_pulse_discovery_Bocas2026'
REGIONS = [dict(r, polygon=[list(p) for p in r['polygon']]) for r in discovery.DEFAULT_REGIONS]
SETTINGS = discovery.Settings(
    seed_threshold=40, boundary_threshold=20,
    min_area=2, preferred_min_area=10, preferred_max_area=2000,
    max_lit_fraction=0.15, max_background_median=8,
    link_radius_px=12, max_missing_frames=1,
    review_count=8, ambiguous_count=2,
    review_min_duration_s=1.0, preferred_duration_s=2.0,
    review_include_tiny=False,
)
FORCE_REPROCESS = False
# Calibration uses a bounded sample of the full clip and only dark ROI frames.
print(pd.DataFrame(REGIONS).to_string(index=False))
print(asdict(SETTINGS))

# August 31 reveals additional vessel areas; use its own saved reference/model.
NIGHT_REFERENCES = {'2026-08-31': PROJECT_ROOT / 'data/selected_videos/physiology/sbs_20260831_232652.mkv'}
NIGHT_CALIBRATION_DIRS = {'2026-08-31': RESULTS_ROOT / 'calibration_aug31'}
NIGHT_REGIONS = {'2026-08-31': [
    dict(id='left_vessel_group', camera='left', polygon=[[820,630],[1570,630],[1570,1080],[820,1080]]),
    dict(id='right_vessel_group', camera='right', polygon=[[2500,620],[3270,620],[3270,1080],[2500,1080]]),
]}


## 1. Inspect search regions and calibrate candidate-light appearance

The reference is 3840 × 1080 at 15 fps. Earlier sampling found many 3–5-pixel recurring components; larger candidates at a 30/255 threshold ranged up to 1,432 pixels. These measurements include apparatus light. Size preferences are therefore ranking penalties, not biological exclusion limits.

Calibration estimates a per-pixel median and recurrent bright cores using accepted ROI samples. The median is used for **ranking intensity changes**, not erasing raw detections. Recurrent cores are provisional fiber associations, not validated fiber identities. Bright frames contribute to the visualization, but not the dark calibration statistics. A whole-video temporal maximum can combine structures seen at different times; verify individual vessel boundaries on a real bright frame before refining the polygons.

The rejection defaults are starting values, not calibrated bright-frame limits across all nights. Inspect rejection rates and representative accepted/rejected frames in the pilot and each new night.

In [ ]:
model = discovery.calibrate(REFERENCE_VIDEO, REGIONS, SETTINGS, RESULTS_ROOT / 'calibration')
display(Image(filename=str(RESULTS_ROOT / 'calibration/regions_reference.jpg'), width=1200))
calibration_components = pd.read_csv(RESULTS_ROOT / 'calibration/calibration_components.csv')
display(calibration_components.groupby('roi_id').area_px.describe(percentiles=[.1,.5,.9,.99]))
# Saved model can also be reopened with discovery.load_calibration(RESULTS_ROOT / 'calibration').

## 2. Run the local pilot

Every frame is inspected inside each region. A low-threshold connected component must contain a brighter seed. CSVs retain small, large, saturated, and fiber-associated candidates with flags. Linking uses a fixed initial centroid with a 12-pixel tolerance, so motion cannot accumulate into a drifting track. Rejected frames immediately break links.

A completed run contains `frames.csv`, `detections.csv`, `candidates.csv`, `sites.csv`, `site_traces.csv`, review clips/contact sheets, and `complete.json`. The completion marker is written only after all outputs succeed; reuse requires matching settings, regions, and calibration arrays.

In [ ]:
start_utc = pd.Timestamp(brightness.parse_utc_start(REFERENCE_VIDEO.name))
night_date = (start_utc.tz_convert(TIMEZONE) - pd.Timedelta(hours=12)).date().isoformat()
pilot_dir = RESULTS_ROOT / night_date / REFERENCE_VIDEO.stem
if FORCE_REPROCESS or not discovery.is_complete(pilot_dir, model, SETTINGS, REFERENCE_VIDEO.name, start_utc):
    pilot_candidates = discovery.scan(REFERENCE_VIDEO, start_utc, model, SETTINGS, pilot_dir)
else:
    pilot_candidates = pd.read_csv(pilot_dir / 'candidates.csv')
print(f'{len(pilot_candidates)} candidate events; these are not confirmed bioluminescence.')
display(pilot_candidates.head(20))
pilot_frames = pd.read_csv(pilot_dir / 'frames.csv')
display(pilot_frames.groupby('roi_id').agg(frames=('frame','size'), accepted_fraction=('accepted','mean'), max_lit_fraction=('lit_fraction','max')))
display(FileLink(str(pilot_dir / 'review/index.html')))

## 3. Review shortlisted segments and illumination decisions

Bioluminescent pulses are expected to hang in place for a few seconds. Review requires at least 1 second (a permissive minimum), favors 2 seconds or longer, and omits components smaller than 10 pixels unless `review_include_tiny=True`. All short/tiny detections remain in CSVs. An empty shortlist is reported explicitly. Scores favor local brightening, coherent duration and position, and less overlap/coincidence with recurrent cores. They are not probabilities. Lower-ranked review examples must also satisfy the duration and size criteria. The original ten reference-clip reviews were reported by the user to show no bioluminescence; they are negative review examples, not confirmed biological pulses. A green circle marks the fixed measurement aperture, not a segmentation boundary. Long events are limited to a 20-second review segment around their peak; full bounds remain in the CSV.

The following cell embeds locally saved images and a clip. Those remain available after a downloaded source video is deleted.

In [ ]:
review_selection = pd.read_csv(pilot_dir / 'review/selection.csv')
if review_selection.empty:
    print('No review candidates meet the current duration and size criteria.')
review_images = [pilot_dir / 'review' / f'{event_id}.jpg' for event_id in review_selection.event_id]
for path in review_images[:3]:
    print(path.stem)
    display(Image(filename=str(path), width=1100))
review_clips = [pilot_dir / 'review' / f'{event_id}.mp4' for event_id in review_selection.event_id]
if review_clips:
    display(Video(filename=str(review_clips[0]), embed=True, width=650))

# Inspect the highest illuminated accepted frame and any rejected frame per ROI.
cap = cv2.VideoCapture(str(REFERENCE_VIDEO))
try:
    for roi_id, group in pilot_frames.groupby('roi_id'):
        rows = []
        accepted = group[group.accepted]
        rejected = group[~group.accepted]
        if len(accepted): rows.append(accepted.loc[accepted.lit_fraction.idxmax()])
        if len(rejected): rows.append(rejected.loc[rejected.lit_fraction.idxmax()])
        r = next(r for r in discovery.compile_regions(REGIONS, model['size']) if r['id'] == roi_id)
        for row in rows:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(row.frame))
            ok, frame = cap.read()
            if not ok: raise RuntimeError('Could not decode diagnostic frame')
            fig, ax = plt.subplots(figsize=(8,5))
            ax.imshow(discovery.crop_gray(frame, r), cmap='gray', vmin=0, vmax=255,
                      extent=[r['x'], r['x']+r['w'], r['y']+r['h'], r['y']])
            ax.set_title(f'{roi_id}: frame {row.frame}; accepted={row.accepted}; lit={row.lit_fraction:.2%}')
            plt.show()
finally:
    cap.release()

## 4. Inspect a fixed site's intensity through light and dark frames

Site IDs group nearby candidate positions **within one video and ROI**. They do not establish animal identity or stereo correspondence. Each fixed 20-pixel-radius aperture is measured throughout the video, including dark frames. Excluded frames have missing intensity, never fabricated zeroes. Saturation fractions identify clipped measurements; apertures may contain neighboring sources, so inspect the overlay.

`duration_s` is the first-to-last detected frame span including both endpoint frames. `missing_frames` records bridged dark/non-detection frames. `left_censored`/`right_censored` mark boundaries next to excluded frames or video edges. Short gaps can be separated by setting `max_missing_frames=0`. No events are automatically joined across recording boundaries.

In [ ]:
eligible_candidates = discovery.select_reviews(pilot_candidates, SETTINGS, model["fps"])
if len(eligible_candidates):
    selected_site = eligible_candidates.iloc[0].site_id  # Change to another site ID to inspect it.
    traces = pd.read_csv(pilot_dir / 'site_traces.csv')
    trace = traces[traces.site_id == selected_site]
    fig, axes = plt.subplots(2,1,figsize=(13,5),sharex=True,constrained_layout=True)
    axes[0].plot(trace.seconds, trace.peak_intensity, lw=.8)
    axes[0].set_ylabel('Aperture peak (0–255)')
    axes[1].plot(trace.seconds, trace.excess_intensity, lw=.8)
    axes[1].set(xlabel='Video time (s)', ylabel='Positive excess sum')
    fig.suptitle(f'Fixed site {selected_site}; missing segments = excluded frames')
    plt.show()
else:
    print('No shortlisted site. All fixed-site traces remain available in site_traces.csv.')


## 5. Discover recordings and choose a small, time-spread batch

This lists Box metadata without downloading recordings. The discovery batch prioritizes 12 evenly spaced recordings from the August 31 Panama observation night, including the following morning, based on the expected biological activity. Set `NIGHTS_TO_PROCESS` to other nights to broaden the search. You can instead supply specific filenames with `VIDEO_NAMES`. Before reusing regions on another night, inspect a sample for camera alignment and vessel coverage; equal dimensions alone do not establish alignment.

In [ ]:
BOX_DATA_RE = re.compile(r"Box\.(?:postStreamData|prefetchedData)\s*=\s*(\{.*?\});", re.DOTALL)
VIDEO_EXTENSIONS = {"avi", "m4v", "mkv", "mov", "mp4", "mpg", "mpeg"}

def _read_url(url, attempts=4):
    headers = {"User-Agent": "Mozilla/5.0", "Accept": "text/html,application/xhtml+xml"}
    for attempt in range(1, attempts + 1):
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read().decode("utf-8")
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as error:
            if attempt == attempts:
                raise RuntimeError(f"Box request failed after {attempts} attempts: {url}") from error
            wait = 2 ** (attempt - 1)
            print(f"Box request failed ({error}); retrying in {wait} seconds...")
            time.sleep(wait)

def night_label(time_panama):
    return (time_panama - pd.Timedelta(hours=NIGHT_BOUNDARY_HOUR)).date().isoformat()

def list_box_videos(shared_url):
    rows, page = [], 1
    while True:
        html = _read_url(f"{shared_url}{'&' if '?' in shared_url else '?'}page={page}")
        candidates = [json.loads(match.group(1)) for match in BOX_DATA_RE.finditer(html)]
        post_data = next((data for data in candidates
                          if "/app-api/enduserapp/shared-folder" in data), None)
        if post_data is None:
            raise RuntimeError("Could not find Box folder metadata. Check that the link is public.")
        folder = post_data["/app-api/enduserapp/shared-folder"]
        rows.extend({"id": int(item["id"]), "name": item["name"],
                     "size_bytes": int(item["itemSize"])}
                    for item in folder["items"]
                    if item.get("type") == "file"
                    and item.get("extension", "").lower() in VIDEO_EXTENSIONS)
        if page >= int(folder["pageCount"]):
            break
        page += 1
    if not rows:
        raise RuntimeError("No video files were found in the shared Box folder.")
    shared_name = post_data["/app-api/enduserapp/shared-item"]["sharedName"]
    table = pd.DataFrame(rows).drop_duplicates("id")
    table["size_GB"] = table["size_bytes"] / 1e9
    table["start_utc"] = table["name"].map(brightness.parse_utc_start)
    bad_names = table.loc[table["start_utc"].isna(), "name"].tolist()
    if bad_names:
        examples = "\n".join(f"  - {name}" for name in bad_names[:10])
        raise ValueError("Could not read a UTC timestamp from these filenames:\n" + examples)
    table["start_utc"] = pd.to_datetime(table["start_utc"], utc=True)
    table["start_panama"] = table["start_utc"].dt.tz_convert(TIMEZONE)
    table["night_date"] = table["start_panama"].map(night_label)
    return table.sort_values(["start_utc", "name"]).reset_index(drop=True), shared_name

def download_box_file(shared_name, file_id, destination, block_size=8 * 1024 * 1024):
    query = urllib.parse.urlencode({"rm": "box_download_shared_file",
                                    "shared_name": shared_name, "file_id": f"f_{file_id}"})
    request = urllib.request.Request(f"https://ucsb.app.box.com/index.php?{query}",
                                     headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120) as response, destination.open("wb") as output:
        total, downloaded = int(response.headers.get("Content-Length", 0)), 0
        while block := response.read(block_size):
            output.write(block)
            downloaded += len(block)
            if total:
                print(f"\rDownloaded {downloaded / total:6.1%}", end="")
    print()

In [ ]:
BOX_URL = 'https://ucsb.box.com/s/dwav1fq9nqys8l1uve1fqdhoez33y4z2'
NIGHTS_TO_PROCESS = ['2026-08-31']  # Panama observation night, including the following morning
VIDEOS_PER_NIGHT = 12
VIDEO_NAMES = []  # Optional exact filenames override evenly spaced selection
videos, shared_name = list_box_videos(BOX_URL)
available_nights = sorted(videos.night_date.unique())
selected_nights = available_nights if NIGHTS_TO_PROCESS is None else NIGHTS_TO_PROCESS
unknown = set(selected_nights) - set(available_nights)
if unknown: raise ValueError(f'Unknown nights: {unknown}')
if VIDEO_NAMES:
    missing = set(VIDEO_NAMES) - set(videos.name)
    if missing: raise ValueError(f'Unknown videos: {missing}')
    selected = videos[videos.name.isin(VIDEO_NAMES)].copy()
else:
    pieces=[]
    for night in selected_nights:
        group=videos[videos.night_date==night].sort_values('start_utc')
        indices=np.unique(np.linspace(0,len(group)-1,min(VIDEOS_PER_NIGHT,len(group)),dtype=int))
        pieces.append(group.iloc[indices])
    selected=pd.concat(pieces,ignore_index=True)
display(selected[['name','night_date','start_panama','size_GB']])
print(f'{len(selected)} videos; {selected.size_GB.sum():.2f} GB total downloads')

## 6. Run the selected discovery batch

Downloads one video at a time; CSVs and cropped review clips are saved before the temporary video is deleted. Incomplete downloads/processing do not create a completion marker. Changing detector settings or calibration invalidates previous results. A failed video is reported and the next one proceeds.

This cell starts the selected downloads when run. August 31 uses its own local reference and expanded regions through `NIGHT_REFERENCES` and `NIGHT_REGIONS`. Other nights fall back to the pilot model; inspect their resulting crops and add a night-specific reference if coverage changes.

In [ ]:
batch_failures=[]
night_models={}
for item in selected.itertuples(index=False):
    if item.night_date not in night_models:
        if item.night_date in NIGHT_REFERENCES:
            night_models[item.night_date] = discovery.calibrate(
                NIGHT_REFERENCES[item.night_date], NIGHT_REGIONS[item.night_date], SETTINGS,
                NIGHT_CALIBRATION_DIRS[item.night_date])
        else:
            night_models[item.night_date] = model
    active_model = night_models[item.night_date]
    destination=RESULTS_ROOT / item.night_date / Path(item.name).stem
    if not FORCE_REPROCESS and discovery.is_complete(destination, active_model, SETTINGS, item.name, item.start_utc):
        print('Already complete:', item.name)
        continue
    print('Processing', item.name, f'({item.size_GB:.3f} GB)')
    try:
        with tempfile.TemporaryDirectory(prefix='physiology_discovery_') as temp_dir:
            local_video=Path(temp_dir)/item.name
            download_box_file(shared_name, item.id, local_video)
            if local_video.stat().st_size != item.size_bytes:
                raise RuntimeError('Downloaded file size does not match Box metadata')
            found=discovery.scan(local_video, item.start_utc, active_model, SETTINGS, destination)
            print(f'Saved {len(found)} candidates and review media to {destination}')
    except Exception as error:
        batch_failures.append(dict(video=item.name,error=str(error)))
        print('FAILED:', item.name, str(error))
display(pd.DataFrame(batch_failures))

## 7. Combined candidate index for separate analysis

Video name + ROI + event/site ID is the unique key. Start/end timestamps refer to detections; frame timestamps allow alternative duration definitions. This table intentionally contains fiber-associated and low-ranked events for later curation. Only completed runs matching the current model/settings are combined.

In [ ]:
tables=[]
for marker in RESULTS_ROOT.glob('*/*/complete.json'):
    info=json.loads(marker.read_text())
    calibration_dir = NIGHT_CALIBRATION_DIRS.get(marker.parent.parent.name)
    active_model = discovery.load_calibration(calibration_dir) if calibration_dir else model
    if info.get('fingerprint') != discovery.fingerprint(active_model, SETTINGS): continue
    table=pd.read_csv(marker.parent/'candidates.csv')
    table['results_directory']=str(marker.parent)
    tables.append(table)
if tables:
    candidate_index=pd.concat(tables,ignore_index=True).sort_values('score',ascending=False)
    candidate_index.to_csv(RESULTS_ROOT/'candidate_index.csv',index=False)
    display(candidate_index.head(50))
    print('Saved', RESULTS_ROOT/'candidate_index.csv')

## 8. Search for overlapping downward trains — August 31

This stage uses all exported detections, not just the old individual-event shortlist. It first isolates sustained brightening above the calibrated baseline: a long-lived hot pixel with a brief reflection is not a multi-second pulse. It then searches for at least three distinct pulse locations with successive onsets and an overall downward trend. Pulse lifetimes may overlap.

Reference: [Hensley et al. (2023), Table 1](https://pmc.ncbi.nlm.nih.gov/articles/PMC10685132/). Mean durations range from 5.711 s for the first pulse to 2.35–4.367 s for later pulses; mean onset intervals are approximately 1.09–1.66 s. These are in situ measurements, not cylinder-specific limits. SEM is not an individual tolerance range. Timing scores are soft; no centimetre constraints or monotonic intensity/duration requirements are imposed.

Initial search limits are deliberately broad: pulse duration 0.7–15 s, onset spacing 0.3–6 s, train span ≤40 s, lateral displacement ≤35 px per step, small upward deviations ≤5 px, and at least 8 px net downward progression. These are adjustable heuristics, not empirical confidence limits. `train_search.COLUMNS` defines provisional vessel columns to avoid linking neighboring cylinders. Inspect the saved column overlay and edit these boundaries if needed.

A second exploratory pass retains tiny components (including potential sensor artifacts) and widens timing/spatial tolerances. Its results require especially careful review. Neither pass can recover pulses missed or merged by the original blob detector. Videos are kept separate, and excluded frames break train evidence.

In [ ]:
import physiologyTrains as train_search
TRAIN_NIGHT = RESULTS_ROOT / '2026-08-31'
TRAIN_SETTINGS = train_search.TrainSettings()
EXPLORATORY_SETTINGS = train_search.TrainSettings(
    min_signal_area=2, min_peak_area=2, min_excess_per_pixel=2,
    min_pulse_s=.5, min_signal_coverage=.5, max_detection_gap_s=.3,
    max_onset_gap_s=8, min_net_down_px=5,
    lateral_tolerance_px=45, upward_tolerance_px=8,
)
train_rows=[]
for marker in sorted(TRAIN_NIGHT.glob('*/complete.json')):
    for mode, configuration in [('trains',TRAIN_SETTINGS), ('trains_exploratory',EXPLORATORY_SETTINGS)]:
        pulses, trains = train_search.analyze_directory(marker.parent, configuration, mode)
        train_rows.append(dict(video=marker.parent.name, mode=mode, pulses=len(pulses), trains=len(trains)))
display(pd.DataFrame(train_rows))
print('Saved pulse-level and train-level CSVs inside each video results directory.')
train_summary = train_search.write_night_summary(TRAIN_NIGHT)
display(train_summary.groupby("mode")[["pulse_candidates", "train_candidates"]].sum())


## 9. Review entire trains

The train gallery is separate from the original 69-clip event gallery. It contains the full train plus three seconds of context on each side, rather than the old 20-second maximum. Pulses are numbered in onset order; green numbers indicate the pulse is active. A zero-train result is reported explicitly.

To render new train clips after tuning, use `train_search.write_train_reviews(source_video, pulses, trains, destination, fps)`. Source videos are needed only for rendering, not for rerunning the CSV-based train search. The combined August 31 gallery below was generated during this run.

In [ ]:
display(FileLink(str(TRAIN_NIGHT / 'trains_index.html')))
if (TRAIN_NIGHT / 'vessel_columns.jpg').exists():
    display(Image(filename=str(TRAIN_NIGHT / 'vessel_columns.jpg'), width=1100))